In [1]:
import os
from dotenv import load_dotenv

#langchain
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import JinaEmbeddings


/var/folders/3z/ng2ls7d967v134v77l0kcyth0000gr/T/ipykernel_1108/356001849.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")
jina_api_key = os.getenv("JINA_API_KEY")



load data

In [4]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")


data ingestion

In [5]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()

print("data loaded successfully")
print(documents)

data loaded successfully
[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of 

langchain document

In [6]:
print(f"total characters in the document: {len(documents[0].page_content)}")

total characters in the document: 2598


split data

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)


[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to

In [8]:
len(chunks)

9

each chunk is a document now (page content and metadata)

In [9]:
print(chunks[0].page_content)
print(chunks[1].page_content)



COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.


embeding data

In [10]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v5-text-small", api_key=jina_api_key)


print("embedding model loaded successfully: ", embeddings_model.model_name)

embedding model loaded successfully:  jina-embeddings-v5-text-small


store data in vector database

In [11]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings_model)


In [12]:
test_query = "How many sick leaves employees get"

## simple similarity search

top_matches = vector_store.similarity_search(test_query, k=2)
print(f"query: {test_query}\n")
for i, match in enumerate(top_matches, start=1):
    print(f"Match {i}: {match.page_content}")

query: How many sick leaves employees get

Match 1: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
Match 2: 7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


Tool

In [13]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def search_hr_policy(question:str)-> str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.

    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

LLM

In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)
llm.model_name

'openai/gpt-oss-120b'

In [15]:
test_response = llm.invoke("hey")

test_response.content

'Hey there! How can I help you today?'

AI Agent : 
LLM - Tool - memory

In [16]:
from langchain.agents import create_agent

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt= """

    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)
print("agent created successfully")



agent created successfully


In [17]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [20]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "How many sick leaves employees get?"
            }
        ]
    }
)

In [21]:
print(response["messages"][-1].content)

Employees are entitled to **10 paid sick days per year**.
